## Аналіз A/B тесту

Маємо проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це класична гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку. У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми проаналізуємо вплив на утримання (retention) гравців. Тобто хочемо зрозуміти чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Змінні в даних наступні:

- userid - унікальний номер, який ідентифікує кожного гравця.
- version - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- sum_gamerounds - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- retention_1 - чи через 1 день після встановлення гравець повернувся і почав грати?
- retention_7 - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/python_labs/chapter 6/cookie_cats.csv')

retention_7_means = df.groupby('version')['retention_7'].mean()

print(retention_7_means)

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


**Гіпотеза**

Версія гри, де ворота знаходяться на рівні 30 (gate_30), ймовірно, забезпечує краще утримання гравців через 7 днів після встановлення гри у порівнянні з версією, де ворота знаходяться на рівні 40 (gate_40).

**Обґрунтування**

Середнє значення показника утримання на 7-й день для версії gate_30 вище (0.190201) порівняно з версією gate_40 (0.182000). Це свідчить про те, що переміщення воріт на 10 рівнів пізніше може негативно впливати на утримання гравців через 7 днів.


2. Перевірте з допомогою z-тесту аналогічно до прикладу в лекції, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для двох вибірок. Виведіть результат у форматі:
```
z statistic: ...
p-value: ...
Довірчий інтервал 95% для групи control: [..., ...]
Довірчий інтервал 95% для групи treatment: [..., ...]
```
де замість `...` - обчислені значення. В якості висновка дайте відповідь на два питання:  
    1. чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
    2. чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  
    
Зверніть увагу, в такому і схожому завданнях ми використовуєм `proportion` Z-тест. Це тому що в нас залежна змінна має бінарне значення (повернеться аби ні користувач, чи клікне або ні користувач в інших ситуаціях - всього два можливих значення в змінної: 0/1, True/False ). Якщо б ми вимірювали скажімо чи є стат. значущою різниця між вагою чоловіків і жінок в певній вибірці, ми б використовувавли функцію `statsmodels.stats.ztest`, бо залежна змінна `вага` є неперервною (тип float, замість типу int чи bool і тільки двох можливих значень).

In [3]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

control_successes = df[df['version'] == 'gate_30']['retention_7'].sum()
control_obs = df[df['version'] == 'gate_30']['retention_7'].count()
treatment_successes = df[df['version'] == 'gate_40']['retention_7'].sum()
treatment_obs = df[df['version'] == 'gate_40']['retention_7'].count()

z_stat, p_val = proportions_ztest([control_successes, treatment_successes], [control_obs, treatment_obs])

control_ci = proportion_confint(control_successes, control_obs, alpha=0.05)
treatment_ci = proportion_confint(treatment_successes, treatment_obs, alpha=0.05)

print(f"z statistic: {z_stat}")
print(f"p-value: {p_val}")
print(f"Довірчий інтервал 95% для групи control: {control_ci}")
print(f"Довірчий інтервал 95% для групи treatment: {treatment_ci}")


z statistic: 3.164358912748191
p-value: 0.001554249975614329
Довірчий інтервал 95% для групи control: (0.18656311652199903, 0.19383956804175934)
Довірчий інтервал 95% для групи treatment: (0.17845430073314686, 0.18554578720019968)


1. Різниця між поведінкою користувачів у різних версіях гри є статистично значущою. Це підтверджується значенням z-статистики (3.1644) та p-значенням (0.0016). Оскільки p-значення < 0.05, ми відхиляємо нульову гіпотезу і визнаємо, що є значуща різниця між групами.

2. Довірчі інтервали для групи control і групи treatment не перетинаються,що ще раз підтверджує, що є статистично значуща різниця між утриманням користувачів у різних версіях гри. Тобто, ми можемо зробити висновок, що версія гри, де ворота знаходяться на рівні 30 (gate_30), має краще утримання користувачів на 7-й день порівняно з версією, де ворота знаходяться на рівні 40 (gate_40).

3. Є ще один тип тестів, який використовується для бінарної метрики як от "зробить юзер дію, чи ні" - тест **Хі-квадрат**. В нього інші гіпотези Н0 і Н1 на відміну від z- та t-тестів. А також цей тест можна використовувати, якщо в нас більше за 2 досліджувані групи, тобто в нас не А/В тест, а А/B/C/D, наприклад.  

В **z- та t-тестах** (які відрізняються тим, що ми в першому не знаємо дисперсію генеральної сукупності, але якщо в нас великий набір даних, то ці два тести дають дуже схожі результати) **ми перевіряємо, чи є різниця у середніх показниках по групам користувачів**.  

А в **тесті Хі-квадрат ми перевіряємо чи є звʼязок між групою користувача і тим, чи він зробить цікаву нам дію**. Це ніби дослідження одного і того самого, але дещо різними способами. Для перевірки, можна виконувати кілька тестів (особливо, якщо один дає якийсь непереконливий результат типу р-значення 0.07 - наче і fail to regect H0 на рівні стат значущості 5%, але цікаво, що скажуть інші тести), тож, зробимо і ми тест хі-квадрат та порівняємо його результат з z-тестом.

Про різницю між тестами можна почитати ще [тут](https://stats.stackexchange.com/a/178860) - це просто пояснення користувача стековерфлоу, але там розумні люди сидять.

Для проведення хі-квадрат тесту скористаємось функцією з `scipy.stats` `chi2_contingency` для обчислення статистики хі-квадрат і р-значення для перевірки конкретної гіпотези. У цю функцію вам треба передати таблицю 2х2: кількість випадків для кожної версії гри і значення `retention_7`.

**Задача**: виконайте тест хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та тим, чи зайде гравець на 7ий день після встановлення гри.
Тут гіпотези наступні
- Н0: значення retention_7 не залежить від версії гри
- Н1: є залежність між версією гри і значенням retention_7

Виведіть p-значення та зробіть висновок.


In [5]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df['version'], df['retention_7'])

chi2, p, dof, expected = chi2_contingency(contingency_table)

print(f"p-value: {p}")

if p < 0.05:
    print("Ми відхиляємо нульову гіпотезу: існує залежність між версією гри та retention_7.")
else:
    print("Ми не можемо відхилити нульову гіпотезу: немає залежності між версією гри та retention_7.")


p-value: 0.0016005742679058301
Ми відхиляємо нульову гіпотезу: існує залежність між версією гри та retention_7.


Згідно проведеного тесту xi-квадрат ми відхиляємо нульову гіпотезу. Отже, існує залежність між версією гри та тим, чи зайде гравець на 7ий день після її встановлення.